# Setup

In [1]:
import duckdb

In [78]:
con = duckdb.connect("gtfs.duckdb")

In [110]:
con.close()

In [17]:
import pandas as pd

# Initial inspect

In [111]:
con.sql("SHOW ALL TABLES").df()

ConnectionException: Connection Error: Connection already closed!

In [15]:
## Assess row count in the static tables:

tables = con.sql("""
    SELECT table_name FROM duckdb_tables() WHERE schema_name = 'raw_gtfs'
""").df()["table_name"].tolist()

query = " UNION ALL ".join(
    f"SELECT '{t}' AS table_name, count(*) AS rows FROM raw_gtfs.{t}"
    for t in tables
)

con.sql(query + " ORDER BY rows DESC").df()


,table_name,rows
0,stop_times,1209051
1,occupancies,518137
2,shapes,96691
3,trips,65947
4,vehicle_boardings,15631
5,stops,1214
6,vehicle_couplings,183
7,routes,137
8,calendar,121
9,vehicle_categories,46


In [105]:
con.sql("""
    SELECT
        feed_timestamp,
        to_timestamp(feed_timestamp) AT TIME ZONE 'Australia/Sydney' AS feed_time_syd
    FROM raw_rt.trip_updates
    LIMIT 1
""").df()

,feed_timestamp,feed_time_syd
0,1789618489,2026-09-17 14:14:49


In [ ]:
con.sql("SELECT * FROM raw_gtfs.trips LIMIT 5;").df()

,route_id,service_id,trip_id,trip_headsign,trip_short_name,direction_id,block_id,shape_id,wheelchair_accessible,vehicle_category_id
0,RTTA_REV,1309.160.100,1--A.1309.160.100.B.8.92115430,Empty Train,,0,,RTTA_REV,0,B8
1,RTTA_REV,1309.160.16,1--A.1309.160.16.B.8.91243188,Empty Train,,0,,RTTA_REV,0,B8
2,RTTA_REV,1309.160.8,1--A.1309.160.8.B.8.92115430,Empty Train,,0,,RTTA_REV,0,B8
3,IWL_2a,1406.153.128,1--A.1406.153.128.B.8.91167462,City Circle via Town Hall,,1,148308,IWL_2a,0,B8
4,IWL_2a,1406.153.2,1--A.1406.153.2.B.8.91167462,City Circle via Town Hall,,1,151202,IWL_2a,0,B8


In [8]:
con.sql("SELECT * FROM raw_rt.trip_updates LIMIT 5;").df()

,feed_timestamp,entity_id,trip_id,route_id,start_date,trip_schedule_relationship,vehicle_id,_loaded_at,stop_sequence,stop_id,arrival_delay,arrival_time,departure_delay,stu_schedule_relationship
0,1789618489,C151.812.150.32.H.8.91118478,C151.812.150.32.H.8.91118478,SCO_1a,,SCHEDULED,,2026-09-17 14:14:52.965992+10:00,NaN,2508162,131.0,0.0,131.0,SCHEDULED
1,1789618489,C151.812.150.32.H.8.91118478,C151.812.150.32.H.8.91118478,SCO_1a,,SCHEDULED,,2026-09-17 14:14:52.965992+10:00,NaN,2515142,131.0,0.0,131.0,SCHEDULED
2,1789618489,C151.812.150.32.H.8.91118478,C151.812.150.32.H.8.91118478,SCO_1a,,SCHEDULED,,2026-09-17 14:14:52.965992+10:00,NaN,2500392,131.0,0.0,131.0,SCHEDULED
3,1789618489,C151.812.150.32.H.8.91118478,C151.812.150.32.H.8.91118478,SCO_1a,,SCHEDULED,,2026-09-17 14:14:52.965992+10:00,NaN,2500402,131.0,0.0,41.0,SCHEDULED
4,1789618489,C151.812.150.32.H.8.91118478,C151.812.150.32.H.8.91118478,SCO_1a,,SCHEDULED,,2026-09-17 14:14:52.965992+10:00,NaN,2500372,41.0,0.0,41.0,SCHEDULED


In [ ]:
## Count how many realtime trips match the schedule (this is the key join):

con.sql("""SELECT
    count(DISTINCT r.trip_id) AS rt_trips,
    count(DISTINCT r.trip_id) FILTER (WHERE t.trip_id IS NULL) AS orphans
FROM raw_rt.trip_updates r
LEFT JOIN raw_gtfs.trips t USING (trip_id);""").df()

### If orphans is very high (more than 50% of rt_trips), 
### the static bundle may not match the realtime feed. Tell me and we will fix it.

,rt_trips,orphans
0,448,0


In [13]:
## See the delay spread in minutes:

con.sql("""
SELECT
    round(arrival_delay / 60.0) AS delay_minutes,
    count(*) AS count
FROM raw_rt.trip_updates
WHERE arrival_delay IS NOT NULL
GROUP BY 1
ORDER BY 1;""").df()

,delay_minutes,count
0,0.0,3210
1,1.0,161
2,2.0,92
3,3.0,23
4,4.0,39
5,5.0,9
6,6.0,38
7,7.0,5
8,8.0,2
9,9.0,1


# Understanding the data

In [51]:
## see the date range your current bundle covers:
con.sql("""
    SELECT min(start_date) AS earliest, max(end_date) AS latest
    FROM raw_gtfs.calendar
""").df()

,earliest,latest
0,20260917,20261016


In [19]:
## An agency is the company that operates the trains. You have two rows.
con.sql("SELECT * FROM raw_gtfs.agency").df()

,agency_id,agency_name,agency_url,agency_timezone,agency_lang,agency_phone
0,NSWTrains,NSW Trains,https://transportnsw.info/regional/,Australia/Sydney,en,
1,SydneyTrains,Sydney Trains,https://transportnsw.info/,Australia/Sydney,en,


In [20]:
## A route is a named line. Each route belongs to one agency.
con.sql("""
    SELECT route_id, agency_id, route_short_name, route_long_name, route_type
    FROM raw_gtfs.routes
    LIMIT 15
""").df()

,route_id,agency_id,route_short_name,route_long_name,route_type
0,APS_1a,SydneyTrains,T8,City Circle to Macarthur via Airport,2
1,APS_1b,SydneyTrains,T8,City Circle to Leppington via Airport,2
2,APS_1c,SydneyTrains,T8,City Circle to Macarthur via Sydenham,2
3,APS_1d,SydneyTrains,T8,City Circle to Macarthur via Sydenham,2
4,APS_1e,SydneyTrains,T8,City Circle to Leppington via Sydenham,2
5,APS_1f,SydneyTrains,T8,City Circle to Leppington via Sydenham,2
6,APS_2a,SydneyTrains,T8,Macarthur to City Circle via Airport,2
7,APS_2b,SydneyTrains,T8,Leppington to City Circle via Airport,2
8,APS_2c,SydneyTrains,T8,Macarthur to City Circle via Sydenham,2
9,APS_2d,SydneyTrains,T8,Macarthur to City Circle via Sydenham,2


In [26]:
## A stop is a physical location where trains stop. This table has a parent-child structure.

con.sql("""
    SELECT *
    FROM raw_gtfs.stops
    LIMIT 5
""").df()

,stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,zone_id,stop_url,location_type,parent_station,stop_timezone,wheelchair_boarding
0,26401,26401,Albury Station,,-36.0840679993,146.924691003,,,1,,,0
1,264086,264086,Albury Station Platform 1,,-36.083941,146.924558,,,0,26401,,0
2,233610,233610,Aberdeen Station,,-32.1671035802,150.892052189,,,1,,,1
3,233621,233621,Aberdeen Station Platform 1,,-32.166969,150.892016,,,0,233610,,0
4,222020,222020,Allawah Station,,-33.9695839058,151.11432998,,,1,,,1


In [27]:
## A parent station is the station itself (e.g. "Central Station"). 
# A child stop is a specific platform at that station (e.g. "Central Station, Platform 16"). 
# The parent_station column on a child row points to the stop_id of its parent.

con.sql("""
    SELECT
        p.stop_name AS station,
        count(*) AS platforms
    FROM raw_gtfs.stops c
    JOIN raw_gtfs.stops p ON c.parent_station = p.stop_id
    WHERE c.parent_station IS NOT NULL AND c.parent_station != ''
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 10
""").df()

,station,platforms
0,Central Station,25
1,Redfern Station,12
2,Strathfield Station,8
3,Blacktown Station,7
4,Homebush Station,6
5,Town Hall Station,6
6,Lidcombe Station,6
7,Burwood Station,6
8,Hornsby Station,5
9,Croydon Station,5


In [31]:
## The calendar tells you which days each service pattern runs.

con.sql("SELECT * FROM raw_gtfs.calendar ORDER BY start_date desc LIMIT 10").df()

,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,2010.102.104,0,1,0,1,1,0,0,20261003,20261016
1,2010.102.108,1,1,0,1,1,0,0,20261003,20261016
2,2010.102.120,0,1,1,1,1,0,0,20261003,20261016
3,2010.102.124,1,1,1,1,1,0,0,20261003,20261016
4,2010.102.128,0,0,0,0,0,1,0,20261003,20261016
5,2010.102.130,0,0,0,0,0,1,1,20261003,20261016
6,2010.102.16,0,0,1,0,0,0,0,20261003,20261016
7,2010.102.2,0,0,0,0,0,0,1,20261003,20261016
8,2010.102.32,0,0,0,1,0,0,0,20261003,20261016
9,2010.102.36,1,0,0,1,0,0,0,20261003,20261016


In [ ]:
## This shows you the most common day patterns. You will likely see weekday-only, weekend-only, and daily patterns.
con.sql("""
    SELECT
        monday, tuesday, wednesday, thursday, friday, saturday, sunday,
        count(*) AS service_patterns
    FROM raw_gtfs.calendar
    GROUP BY 1,2,3,4,5,6,7
    ORDER BY 8 DESC
    """).df()

,monday,tuesday,wednesday,thursday,friday,saturday,sunday,service_patterns
0,0,0,1,0,0,0,0,4
1,0,1,0,1,0,0,0,4
2,1,1,1,1,1,0,0,4
3,0,1,0,0,1,0,0,4
4,1,1,0,1,1,0,0,4
5,0,0,1,1,0,0,0,4
6,0,1,1,0,1,0,0,4
7,0,1,1,1,1,0,0,4
8,1,1,0,1,0,0,0,4
9,1,0,1,1,0,0,0,4


In [ ]:
## A trip is one train journey. It connects a route, a service pattern, and a direction.

con.sql("""
    SELECT trip_id, route_id, service_id, direction_id, trip_headsign, shape_id
    FROM raw_gtfs.trips
    LIMIT 10""").df()

## trip_headsign is the destination shown on the front of the train (e.g. "Hornsby via Strathfield").
## direction_id is 0 or 1 — outbound or inbound. The GTFS spec does not define which is which. You must work that out from the data.
## shape_id links to the shapes table, which holds the GPS path the train follows. Useful for maps, not needed for your project.

,trip_id,route_id,service_id,direction_id,trip_headsign,shape_id
0,1--A.1309.160.100.B.8.92115430,RTTA_REV,1309.160.100,0,Empty Train,RTTA_REV
1,1--A.1309.160.16.B.8.91243188,RTTA_REV,1309.160.16,0,Empty Train,RTTA_REV
2,1--A.1309.160.8.B.8.92115430,RTTA_REV,1309.160.8,0,Empty Train,RTTA_REV
3,1--A.1406.153.128.B.8.91167462,IWL_2a,1406.153.128,1,City Circle via Town Hall,IWL_2a
4,1--A.1406.153.2.B.8.91167462,IWL_2a,1406.153.2,1,City Circle via Town Hall,IWL_2a
5,1--A.2010.102.124.B.8.91040058,RTTA_REV,2010.102.124,0,Empty Train,RTTA_REV
6,1--A.2010.102.128.B.8.91040060,IWL_2a,2010.102.128,1,City Circle via Town Hall,IWL_2a
7,1--A.2010.102.2.B.8.91040060,IWL_2a,2010.102.2,1,City Circle via Town Hall,IWL_2a
8,1--A.812.150.112.B.8.91118315,RTTA_REV,812.150.112,0,Empty Train,RTTA_REV
9,1--A.812.150.4.B.8.91118315,RTTA_REV,812.150.4,0,Empty Train,RTTA_REV


In [35]:
## This query shows how many trips each route has:

con.sql("""
    SELECT r.route_short_name, count(*) AS trips
    FROM raw_gtfs.trips t
    JOIN raw_gtfs.routes r USING (route_id)
    GROUP BY 1
    ORDER BY 2 DESC
""").df()

,route_short_name,trips
0,T1,10334
1,,9113
2,T8,7864
3,T2,6510
4,T9,6277
5,T4,6107
6,T7,3565
7,SCO,3303
8,CCN,2556
9,T6,2521


In [ ]:
'''A route is not one train. A route is the line itself — like the T1 North Shore Line drawn on the network map. Every individual train that runs on that line is a separate trip.

Think of it this way. The T1 line might run a train every 5 minutes during peak hours, every 10 minutes off-peak, and every 15 minutes late at night. That adds up to roughly 150–200 trips per day. Multiply by the number of days in the timetable period (often 30–60 days), and you get thousands of trips.'''

### This query shows the breakdown:

con.sql("""
    SELECT
        r.route_short_name,
        count(DISTINCT t.trip_id) AS total_trips,
        count(DISTINCT c.service_id) AS service_patterns,
        round(count(DISTINCT t.trip_id) / count(DISTINCT c.service_id)) AS approx_trips_per_pattern
    FROM raw_gtfs.trips t
    JOIN raw_gtfs.routes r USING (route_id)
    JOIN raw_gtfs.calendar c USING (service_id)
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 10
""").df()

## The approx_trips_per_pattern column gives you a rough idea of how many trains run the line each day.

,route_short_name,total_trips,service_patterns,approx_trips_per_pattern
0,T1,10334,70,148.0
1,,9113,102,89.0
2,T8,7864,67,117.0
3,T2,6510,69,94.0
4,T9,6277,44,143.0
5,T4,6107,61,100.0
6,T7,3565,27,132.0
7,SCO,3303,47,70.0
8,CCN,2556,61,42.0
9,T6,2521,34,74.0


In [ ]:
## stop_times is the largest and most important table. Each row says: "Trip X arrives at stop Y at time Z."

con.sql("""
    SELECT trip_id, arrival_time, departure_time, stop_id, stop_sequence, pickup_type, drop_off_type
    FROM raw_gtfs.stop_times
    ORDER BY trip_id asc
    LIMIT 10
""").df()

## stop_sequence is the order. Sequence 1 is the first stop, 2 is the second, and so on.
## arrival_time and departure_time are strings like "07:32:00". They can go past midnight — "25:10:00" means 1:10am on the next calendar day but still belongs to the previous service day.
## pickup_type and drop_off_type tell you whether passengers can board or alight. 0 means yes.

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,pickup_type,drop_off_type
0,1--A.1309.160.100.B.8.92115430,05:26:00,05:26:00,2142324,8,1,1
1,1--A.1309.160.100.B.8.92115430,05:28:30,05:28:30,2150404,15,1,1
2,1--A.1309.160.100.B.8.92115430,05:30:00,05:43:30,2150414,18,1,1
3,1--A.1309.160.16.B.8.91243188,05:26:00,05:26:00,2142324,8,1,1
4,1--A.1309.160.16.B.8.91243188,05:30:00,05:30:00,2150404,15,1,1
5,1--A.1309.160.16.B.8.91243188,05:32:00,05:43:30,2150414,18,1,1
6,1--A.1309.160.8.B.8.92115430,05:26:00,05:26:00,2142324,8,1,1
7,1--A.1309.160.8.B.8.92115430,05:28:30,05:28:30,2150404,15,1,1
8,1--A.1309.160.8.B.8.92115430,05:30:00,05:43:30,2150414,18,1,1
9,1--A.1406.153.128.B.8.91167462,04:49:00,04:49:00,2144243,8,1,1


In [48]:
## This query shows a single trip's full journey — every stop in order:

con.sql("""
    SELECT
        st.stop_sequence,
        st.arrival_time,
        st.departure_time,
        s.stop_name
    FROM raw_gtfs.stop_times st
    JOIN raw_gtfs.stops s USING (stop_id)
    WHERE st.trip_id = (SELECT trip_id FROM raw_gtfs.trips ORDER BY random() LIMIT 1)
    ORDER BY CAST(st.stop_sequence AS INTEGER)
""").df()

,stop_sequence,arrival_time,departure_time,stop_name
0,1,12:25:01,12:50:01,Central Station Platform 6
1,19,12:53:00,12:53:00,Redfern Station Platform 2
2,34,12:59:30,12:59:30,Croydon Station Platform 1
3,35,13:00:24,13:00:24,Burwood Station Platform 2
4,37,13:02:00,13:03:00,Strathfield Station Platform 3
5,45,13:05:36,13:05:36,North Strathfield Station Platform 2
6,46,13:06:30,13:06:30,Concord West Station Platform 3
7,50,13:07:54,13:07:54,Rhodes Station Platform 2
8,55,13:08:54,13:08:54,Meadowbank Station Platform 2
9,57,13:09:30,13:09:30,West Ryde Station Platform 3


In [ ]:
## raw_rt.trip_updates (realtime)
## This is the snapshot you collected. 
## Each row says: "Right now, trip X is predicted to arrive at stop Y with a delay of Z seconds."

con.sql("""
    SELECT trip_id, stop_id, stop_sequence, arrival_delay, departure_delay,
           stu_schedule_relationship, feed_timestamp, _loaded_at
    FROM raw_rt.trip_updates
    LIMIT 10
""").df()

## arrival_delay is in seconds. A value of 60 means one minute late. A value of -30 means 30 seconds early.
## feed_timestamp is the time the feed was generated (POSIX/Unix timestamp, UTC).
## stu_schedule_relationship tells you the status of this stop. SCHEDULED is normal. SKIPPED means the train will not stop there.

,trip_id,stop_id,stop_sequence,arrival_delay,departure_delay,stu_schedule_relationship,feed_timestamp,_loaded_at
0,C151.812.150.32.H.8.91118478,2508162,NaN,131.0,131.0,SCHEDULED,1789618489,2026-09-17 14:14:52.965992+10:00
1,C151.812.150.32.H.8.91118478,2515142,NaN,131.0,131.0,SCHEDULED,1789618489,2026-09-17 14:14:52.965992+10:00
2,C151.812.150.32.H.8.91118478,2500392,NaN,131.0,131.0,SCHEDULED,1789618489,2026-09-17 14:14:52.965992+10:00
3,C151.812.150.32.H.8.91118478,2500402,NaN,131.0,41.0,SCHEDULED,1789618489,2026-09-17 14:14:52.965992+10:00
4,C151.812.150.32.H.8.91118478,2500372,NaN,41.0,41.0,SCHEDULED,1789618489,2026-09-17 14:14:52.965992+10:00
5,C151.812.150.32.H.8.91118478,2526171,NaN,41.0,41.0,SCHEDULED,1789618489,2026-09-17 14:14:52.965992+10:00
6,C151.812.150.32.H.8.91118478,2526161,NaN,41.0,41.0,SCHEDULED,1789618489,2026-09-17 14:14:52.965992+10:00
7,C151.812.150.32.H.8.91118478,2530232,NaN,41.0,41.0,SCHEDULED,1789618489,2026-09-17 14:14:52.965992+10:00
8,C151.812.150.32.H.8.91118478,2527162,NaN,41.0,0.0,SCHEDULED,1789618489,2026-09-17 14:14:52.965992+10:00
9,C151.812.150.32.H.8.91118478,2529201,NaN,0.0,0.0,SCHEDULED,1789618489,2026-09-17 14:14:52.965992+10:00


In [62]:
con.sql("""
    SELECT
        CAST(st.stop_sequence AS INTEGER) AS seq,
        s.stop_name,
        st.arrival_time AS scheduled,
        rt.arrival_delay AS delay_seconds
    FROM raw_rt.trip_updates rt
    JOIN raw_gtfs.stop_times st
        ON rt.trip_id = st.trip_id
        AND rt.stop_id = st.stop_id
    JOIN raw_gtfs.stops s ON st.stop_id = s.stop_id
    WHERE rt.trip_id = '14-G.812.150.48.B.8.91116705'
    ORDER BY CAST(st.stop_sequence AS INTEGER)
""").df()

# This is an important finding for your project. It means:

# Your join key is trip_id + stop_id (not stop_sequence).
# You should document this in your dbt model as a design decision.
# You should add a data quality test that checks whether stop_sequence is ever populated in the realtime data. If it is sometimes populated and sometimes not, that is worth noting.

,seq,stop_name,scheduled,delay_seconds
0,0,Central Station Platform 17,14:19:31,0.0
1,6,Town Hall Station Platform 6,14:23:18,0.0
2,8,Wynyard Station Platform 6,14:26:00,0.0
3,12,Circular Quay Station Platform 2,14:29:00,0.0
4,15,St James Station Platform 2,14:32:18,0.0
5,16,Museum Station Platform 2,14:34:12,0.0
6,20,Central Station Platform 23,14:37:00,0.0
7,25,Green Square Station Platform 2,14:41:18,0.0
8,26,Mascot Station Platform 2,14:45:00,0.0
9,28,Domestic Station Platform 2,14:48:00,0.0


In [65]:
## This should give you a clean table of station names with scheduled times and delays.

con.sql("""
    SELECT
        CAST(st.stop_sequence AS INTEGER) AS seq,
        parent.stop_name AS station,
        st.arrival_time AS scheduled,
        rt.arrival_delay AS delay_seconds
    FROM raw_rt.trip_updates rt
    JOIN raw_gtfs.stop_times st
        ON rt.trip_id = st.trip_id
        AND rt.stop_id = st.stop_id
    JOIN raw_gtfs.stops platform ON st.stop_id = platform.stop_id
    JOIN raw_gtfs.stops parent ON platform.parent_station = parent.stop_id
    WHERE rt.trip_id = '14-G.812.150.48.B.8.91116705'
    ORDER BY CAST(st.stop_sequence AS INTEGER)
""").df()

,seq,station,scheduled,delay_seconds
0,0,Central Station,14:19:31,0.0
1,6,Town Hall Station,14:23:18,0.0
2,8,Wynyard Station,14:26:00,0.0
3,12,Circular Quay Station,14:29:00,0.0
4,15,St James Station,14:32:18,0.0
5,16,Museum Station,14:34:12,0.0
6,20,Central Station,14:37:00,0.0
7,25,Green Square Station,14:41:18,0.0
8,26,Mascot Station,14:45:00,0.0
9,28,Domestic Airport Station,14:48:00,0.0


# Inspecting the poller snapshot output
I.e. from test_poller.py

In [83]:
## Step 1 — Find trips that appear in many snapshots:

con.sql("""
    SELECT trip_id, count(DISTINCT snapshot_ts) AS snapshots, count(*) AS rows
    FROM raw_rt.poller_test
    GROUP BY trip_id
    HAVING count(DISTINCT snapshot_ts) > 10
    ORDER BY snapshots DESC
    LIMIT 10
""").df()

,trip_id,snapshots,rows
0,73AZ.812.150.48.M.4.91117208,20,56
1,55AL.812.150.60.M.4.91116957,20,56
2,157N.812.150.32.A.8.91116630,20,108
3,8--T.812.150.48.B.8.91115255,20,382
4,28-M.812.150.48.B.8.91115648,20,491
5,601E.812.150.60.T.8.91118734,20,20
6,W667.812.150.44.D.10.91117900,20,20
7,89-M.812.150.32.A.8.91117173,20,224
8,619N.812.150.120.T.8.91114574,20,420
9,128S.812.150.112.A.8.91115627,20,138


In [89]:
selected_trip_id = '8--T.812.150.48.B.8.91115255'

In [92]:
## Step 2 — See all stations and delays for that trip, across all snapshots:

con.sql(f"""
    SELECT
        to_timestamp(p.snapshot_ts) AT TIME ZONE 'Australia/Sydney' AS snapshot_time,
        parent.stop_name AS station,
        st.arrival_time AS scheduled,
        p.arrival_delay AS delay_seconds
    FROM raw_rt.poller_test p
    JOIN raw_gtfs.stop_times st
        ON p.trip_id = st.trip_id AND p.stop_id = st.stop_id
    JOIN raw_gtfs.stops platform ON p.stop_id = platform.stop_id
    JOIN raw_gtfs.stops parent ON platform.parent_station = parent.stop_id
    WHERE p.trip_id = '{selected_trip_id}'
    --ORDER BY CAST(st.stop_sequence AS INTEGER), snapshot_time
    ORDER BY RANDOM()
    LIMIT 30
""").df()

,snapshot_time,station,scheduled,delay_seconds
0,2026-09-17 18:31:35,St James Station,18:26:18,561.0
1,2026-09-17 18:23:50,Central Station,18:31:00,492.0
2,2026-09-17 18:25:50,Bexley North Station,18:54:30,510.0
3,2026-09-17 18:30:05,Bardwell Park Station,18:52:18,498.0
4,2026-09-17 18:31:05,Narwee Station,19:01:42,538.0
5,2026-09-17 18:32:05,Beverly Hills Station,18:59:30,492.0
6,2026-09-17 18:27:20,Mascot Station,18:39:00,468.0
7,2026-09-17 18:30:05,Green Square Station,18:35:18,498.0
8,2026-09-17 18:28:20,Mascot Station,18:39:00,519.0
9,2026-09-17 18:29:35,Bexley North Station,18:54:30,489.0


In [95]:
con.sql(f"""
    SELECT
        CAST(st.stop_sequence AS INTEGER) AS seq,
        parent.stop_name AS station,
        st.arrival_time AS scheduled
    FROM raw_gtfs.stop_times st
    JOIN raw_gtfs.stops platform ON st.stop_id = platform.stop_id
    JOIN raw_gtfs.stops parent ON platform.parent_station = parent.stop_id
    WHERE st.trip_id = '{selected_trip_id}'
    ORDER BY CAST(st.stop_sequence AS INTEGER)
""").df()

,seq,station,scheduled
0,0,Central Station,18:13:31
1,6,Town Hall Station,18:17:18
2,8,Wynyard Station,18:20:00
3,12,Circular Quay Station,18:23:00
4,15,St James Station,18:26:18
5,16,Museum Station,18:28:12
6,20,Central Station,18:31:00
7,25,Green Square Station,18:35:18
8,26,Mascot Station,18:39:00
9,28,Domestic Airport Station,18:42:00


In [101]:
selected_stop_name = 'Wynyard Station'

In [102]:
## Step 3 — Isolate one station to see its delay values change over time:

con.sql(f"""
    SELECT
        to_timestamp(p.snapshot_ts) AT TIME ZONE 'Australia/Sydney' AS snapshot_time,
        parent.stop_name AS station,
        st.arrival_time AS scheduled,
        p.arrival_delay AS delay_seconds
    FROM raw_rt.poller_test p
    JOIN raw_gtfs.stop_times st
        ON p.trip_id = st.trip_id AND p.stop_id = st.stop_id
    JOIN raw_gtfs.stops platform ON p.stop_id = platform.stop_id
    JOIN raw_gtfs.stops parent ON platform.parent_station = parent.stop_id
    WHERE p.trip_id = '{selected_trip_id}'
      AND parent.stop_name = '{selected_stop_name}'
    ORDER BY snapshot_time
""").df()

,snapshot_time,station,scheduled,delay_seconds
0,2026-09-17 18:23:50,Wynyard Station,18:20:00,522.0
1,2026-09-17 18:24:20,Wynyard Station,18:20:00,537.0
2,2026-09-17 18:24:50,Wynyard Station,18:20:00,543.0
3,2026-09-17 18:25:20,Wynyard Station,18:20:00,570.0
4,2026-09-17 18:25:50,Wynyard Station,18:20:00,570.0
5,2026-09-17 18:26:20,Wynyard Station,18:20:00,590.0
6,2026-09-17 18:26:50,Wynyard Station,18:20:00,590.0
7,2026-09-17 18:27:20,Wynyard Station,18:20:00,528.0
8,2026-09-17 18:27:50,Wynyard Station,18:20:00,549.0
9,2026-09-17 18:28:20,Wynyard Station,18:20:00,579.0


In [104]:
## Run this query to see all stops on the trip, with only their last snapshot. This gives you the "estimated actual" for every stop the train had passed during your collection window:

con.sql(f"""
    WITH last_snapshot AS (
        SELECT
            p.trip_id,
            p.stop_id,
            p.arrival_delay,
            p.snapshot_ts,
            ROW_NUMBER() OVER (
                PARTITION BY p.trip_id, p.stop_id
                ORDER BY p.snapshot_ts DESC
            ) AS rn
        FROM raw_rt.poller_test p
        WHERE p.trip_id = '{selected_trip_id}'
    )
    SELECT
        CAST(st.stop_sequence AS INTEGER) AS seq,
        parent.stop_name AS station,
        st.arrival_time AS scheduled,
        ls.arrival_delay AS last_delay_seconds,
        to_timestamp(ls.snapshot_ts) AT TIME ZONE 'Australia/Sydney' AS last_seen
    FROM last_snapshot ls
    JOIN raw_gtfs.stop_times st
        ON ls.trip_id = st.trip_id AND ls.stop_id = st.stop_id
    JOIN raw_gtfs.stops platform ON ls.stop_id = platform.stop_id
    JOIN raw_gtfs.stops parent ON platform.parent_station = parent.stop_id
    WHERE ls.rn = 1
    ORDER BY CAST(st.stop_sequence AS INTEGER)
""").df()

,seq,station,scheduled,last_delay_seconds,last_seen
0,0,Central Station,18:13:31,583.0,2026-09-17 18:24:20
1,6,Town Hall Station,18:17:18,558.0,2026-09-17 18:27:50
2,8,Wynyard Station,18:20:00,549.0,2026-09-17 18:30:05
3,12,Circular Quay Station,18:23:00,522.0,2026-09-17 18:32:35
4,15,St James Station,18:26:18,567.0,2026-09-17 18:33:35
5,16,Museum Station,18:28:12,567.0,2026-09-17 18:33:35
6,20,Central Station,18:31:00,567.0,2026-09-17 18:33:35
7,25,Green Square Station,18:35:18,537.0,2026-09-17 18:33:35
8,26,Mascot Station,18:39:00,537.0,2026-09-17 18:33:35
9,28,Domestic Airport Station,18:42:00,537.0,2026-09-17 18:33:35


In [ ]:
'''Exactly right. You can tell the difference. Look at the last_seen column. The first four stops each have a different last_seen time — they were dropped from the feed one by one as the train passed them. Every stop from St James onwards has the same last_seen time of 18:33:35 — that is your final snapshot. Those stops were still in the feed when collection ended, meaning the train had not reached them yet.'''

# The rule is simple: if last_seen equals the timestamp of the last snapshot in the collection, the train had not yet passed that stop.

# Add a column to make this explicit:

con.sql(f"""
    WITH last_snapshot AS (
        SELECT
            p.trip_id,
            p.stop_id,
            p.arrival_delay,
            p.snapshot_ts,
            ROW_NUMBER() OVER (
                PARTITION BY p.trip_id, p.stop_id
                ORDER BY p.snapshot_ts DESC
            ) AS rn
        FROM raw_rt.poller_test p
        WHERE p.trip_id = '{selected_trip_id}'
    ),
    collection_end AS (
        SELECT max(snapshot_ts) AS max_ts
        FROM raw_rt.poller_test
    )
    SELECT
        CAST(st.stop_sequence AS INTEGER) AS seq,
        parent.stop_name AS station,
        st.arrival_time AS scheduled,
        ls.arrival_delay AS last_delay_seconds,
        to_timestamp(ls.snapshot_ts) AT TIME ZONE 'Australia/Sydney' AS last_seen,
        CASE
            WHEN ls.snapshot_ts = ce.max_ts THEN 'prediction'
            ELSE 'estimated actual'
        END AS confidence
    FROM last_snapshot ls
    CROSS JOIN collection_end ce
    JOIN raw_gtfs.stop_times st
        ON ls.trip_id = st.trip_id AND ls.stop_id = st.stop_id
    JOIN raw_gtfs.stops platform ON ls.stop_id = platform.stop_id
    JOIN raw_gtfs.stops parent ON platform.parent_station = parent.stop_id
    WHERE ls.rn = 1
    ORDER BY CAST(st.stop_sequence AS INTEGER)
""").df()

,seq,station,scheduled,last_delay_seconds,last_seen,confidence
0,0,Central Station,18:13:31,583.0,2026-09-17 18:24:20,estimated actual
1,6,Town Hall Station,18:17:18,558.0,2026-09-17 18:27:50,estimated actual
2,8,Wynyard Station,18:20:00,549.0,2026-09-17 18:30:05,estimated actual
3,12,Circular Quay Station,18:23:00,522.0,2026-09-17 18:32:35,estimated actual
4,15,St James Station,18:26:18,567.0,2026-09-17 18:33:35,prediction
5,16,Museum Station,18:28:12,567.0,2026-09-17 18:33:35,prediction
6,20,Central Station,18:31:00,567.0,2026-09-17 18:33:35,prediction
7,25,Green Square Station,18:35:18,537.0,2026-09-17 18:33:35,prediction
8,26,Mascot Station,18:39:00,537.0,2026-09-17 18:33:35,prediction
9,28,Domestic Airport Station,18:42:00,537.0,2026-09-17 18:33:35,prediction
